# RAG Sufficient Context: Selective Generation Pipeline

**Explaining and Reducing Hallucinations in RAG via Sufficient Context**

Inspired by: *Sufficient Context: A New Lens on Systems Retrieval Augmented Generation* (ICLR 2025)

Authors: Aleksandr Gavkovskii, Ilya Maksimov, Karim Zakirov

---

This notebook runs the full pipeline end-to-end:
1. Setup & install dependencies
2. Load HotPotQA dataset
3. BM25 retrieval & context construction
4. LLM generation (LLaMA-3.1-8B-Instruct)
5. Evaluation (EM/F1, categorization)
6. Sufficient context autorater
7. Selective generation gate
8. Visualization & analysis

## 1. Setup

In [ ]:
# Clone the repository and install dependencies
!git clone https://github.com/SachaYT1/rag-sufficient-context.git
%cd rag-sufficient-context
!pip install -q -r requirements.txt

In [ ]:
# HuggingFace login (required for LLaMA-3.1-8B, a gated model)
# 1. Go to https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct and accept the license
# 2. Create a READ token at https://huggingface.co/settings/tokens
# 3. Paste it below when prompted
from huggingface_hub import login
login()

In [ ]:
# Imports
import os
import json
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint

from src.utils import load_config, cache_results, load_cached
from src.retrieval import load_hotpotqa, build_retrieval_pipeline
from src.generation import load_model, generate_answers_batch
from src.evaluation import evaluate_all, categorize_response
from src.autorater import rate_all_examples
from src.confidence import estimate_confidence_batch
from src.gate import (
    prepare_features, train_gate,
    compute_selective_curves, plot_accuracy_coverage,
    plot_sufficiency_breakdown,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load configuration
config = load_config("configs/default.yaml")
pprint(config)

## 2. Load Dataset

In [ ]:
# Load HotPotQA dev subset
examples = load_hotpotqa(
    split=config["dataset"]["split"],
    num_examples=config["dataset"]["num_examples"],
    seed=config["dataset"]["seed"],
)
print(f"Loaded {len(examples)} examples")
print(f"\nExample:")
print(f"  Question: {examples[0]['question']}")
print(f"  Answer: {examples[0]['answer']}")
print(f"  Type: {examples[0]['type']}")
print(f"  Num passages: {len(examples[0]['passages'])}")

## 3. BM25 Retrieval & Context Construction

In [ ]:
# Check for cached retrieval results
retrieval_cache = "data/retrieval_results.json"
cached = load_cached(retrieval_cache)

if cached is not None:
    examples_with_context = cached
    print(f"Loaded {len(examples_with_context)} cached retrieval results")
else:
    # Load tokenizer for truncation
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(config["generation"]["model"])

    examples_with_context = build_retrieval_pipeline(
        examples,
        top_k=config["retrieval"]["top_k"],
        max_context_tokens=config["retrieval"]["max_context_tokens"],
        tokenizer=tokenizer,
    )
    cache_results(examples_with_context, retrieval_cache)
    print(f"Retrieved contexts for {len(examples_with_context)} examples (cached)")

# Show example context
print(f"\nContext length (chars): {len(examples_with_context[0]['context'])}")
print(f"Context preview: {examples_with_context[0]['context'][:300]}...")

## 4. LLM Generation

In [ ]:
# Load model
model, tokenizer = load_model(
    model_name=config["generation"]["model"],
    device="auto",
)
print(f"Model loaded: {config['generation']['model']}")
print(f"Device: {next(model.parameters()).device}")

In [ ]:
# Generate answers
generation_cache = "data/generation_results.json"
cached = load_cached(generation_cache)

if cached is not None:
    examples_with_answers = cached
    print(f"Loaded {len(examples_with_answers)} cached generation results")
else:
    examples_with_answers = generate_answers_batch(
        examples_with_context,
        model, tokenizer,
        max_new_tokens=config["generation"]["max_new_tokens"],
        temperature=config["generation"]["temperature"],
    )
    cache_results(examples_with_answers, generation_cache)
    print(f"Generated answers for {len(examples_with_answers)} examples (cached)")

# Show example
ex = examples_with_answers[0]
print(f"\nQuestion: {ex['question']}")
print(f"Gold answer: {ex['answer']}")
print(f"Prediction: {ex['prediction']}")
print(f"Confidence: {ex['confidence']:.3f}")

## 5. Evaluation

In [ ]:
# Evaluate all examples
eval_results = evaluate_all(examples_with_answers)
metrics = eval_results["metrics"]

print("=" * 50)
print("BASELINE EVALUATION RESULTS")
print("=" * 50)
print(f"Total examples: {metrics['total']}")
print(f"\nCategories:")
print(f"  Correct:      {metrics['correct']:>4d} ({metrics['correct_rate']:.1%})")
print(f"  Abstain:      {metrics['abstain']:>4d} ({metrics['abstain_rate']:.1%})")
print(f"  Hallucinate:  {metrics['hallucinate']:>4d} ({metrics['hallucinate_rate']:.1%})")
print(f"\nMetrics:")
print(f"  Mean EM: {metrics['mean_em']:.3f}")
print(f"  Mean F1: {metrics['mean_f1']:.3f}")

In [ ]:
# Add category labels to examples
for ex, per_ex in zip(examples_with_answers, eval_results["per_example"]):
    ex["category"] = per_ex["category"]
    ex["em"] = per_ex["em"]
    ex["f1"] = per_ex["f1"]

## 6. Sufficient Context Autorater

In [ ]:
# Rate context sufficiency
autorater_cache = "data/autorater_results.json"
cached = load_cached(autorater_cache)

if cached is not None:
    examples_with_sufficiency = cached
    print(f"Loaded {len(examples_with_sufficiency)} cached autorater results")
else:
    examples_with_sufficiency = rate_all_examples(
        examples_with_answers,
        model, tokenizer,
        chunk_size_tokens=config["autorater"]["chunk_size_tokens"],
        aggregation=config["autorater"]["aggregation"],
    )
    cache_results(examples_with_sufficiency, autorater_cache)
    print(f"Rated sufficiency for {len(examples_with_sufficiency)} examples (cached)")

# Summary
n_sufficient = sum(1 for ex in examples_with_sufficiency if ex.get("sufficient"))
n_total = len(examples_with_sufficiency)
print(f"\nSufficient: {n_sufficient}/{n_total} ({n_sufficient/n_total:.1%})")
print(f"Insufficient: {n_total - n_sufficient}/{n_total} ({(n_total - n_sufficient)/n_total:.1%})")

## 7. Selective Generation Gate

In [ ]:
# Prepare features and train gate
X, y = prepare_features(examples_with_sufficiency)
print(f"Training data: {X.shape[0]} examples (excluding abstentions)")
print(f"Positive (correct): {y.sum():.0f}, Negative (hallucinate): {(1-y).sum():.0f}")

# Train logistic regression gate
gate = train_gate(X, y)
print(f"\nGate coefficients:")
print(f"  confidence weight: {gate.coef_[0][0]:.3f}")
print(f"  sufficiency weight: {gate.coef_[0][1]:.3f}")
print(f"  intercept: {gate.intercept_[0]:.3f}")

In [ ]:
# Compute selective accuracy-coverage curves
curves = compute_selective_curves(
    examples_with_sufficiency,
    gate=gate,
    threshold_steps=config["gate"]["threshold_steps"],
)

# Save curves
cache_results(curves, "results/selective_curves.json")
print("Curves computed and saved")

## 8. Visualization & Analysis

In [ ]:
# Plot 1: Selective Accuracy vs. Coverage
plot_accuracy_coverage(curves, save_path="results/accuracy_coverage.png")

In [ ]:
# Plot 2: Sufficiency breakdown
plot_sufficiency_breakdown(examples_with_sufficiency, save_path="results/sufficiency_breakdown.png")

In [ ]:
# Summary table
import pandas as pd

# Create summary dataframe
summary_data = []
for ex in examples_with_sufficiency:
    summary_data.append({
        "question": ex["question"][:80],
        "category": ex.get("category", "unknown"),
        "sufficient": ex.get("sufficient", False),
        "confidence": ex.get("confidence", 0.0),
        "em": ex.get("em", 0.0),
        "f1": ex.get("f1", 0.0),
    })

df = pd.DataFrame(summary_data)

# Cross-tabulation
print("Category breakdown by sufficiency:")
print(pd.crosstab(df["sufficient"], df["category"], margins=True))
print()

# Mean confidence by category
print("Mean confidence by category:")
print(df.groupby("category")["confidence"].mean().round(3))

In [ ]:
# Save all results
final_results = {
    "metrics": metrics,
    "n_sufficient": n_sufficient,
    "n_insufficient": n_total - n_sufficient,
    "gate_coefficients": {
        "confidence_weight": float(gate.coef_[0][0]),
        "sufficiency_weight": float(gate.coef_[0][1]),
        "intercept": float(gate.intercept_[0]),
    },
}
cache_results(final_results, "results/final_metrics.json")
print("All results saved to results/")
print("\nDone! Pipeline complete.")